# Jointure trajectoires × météo

---
## 1 - Configuration

In [1]:
import os, re, gc, warnings
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import polars as pl

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from scipy.interpolate import RegularGridInterpolator

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

try:
    import xarray as xr
    XARRAY_OK = True
except ImportError:
    XARRAY_OK = False
    print("xarray n'est pas installé.")

try:
    import cfgrib
    CFGRIB_OK = True
except ImportError:
    CFGRIB_OK = False
    print("cfgrib n'est pas installé.")

try:
    import dask
    DASK_OK = True
except ImportError:
    DASK_OK = False
    print("dask n'est pas installé.")

try:
    import pyarrow
    PYARROW_OK = True
except ImportError:
    PYARROW_OK = False
    print("pyarrow n'est pas installé (nécessaire pour l'export Parquet zstd).")

print(f"xarray={XARRAY_OK}  cfgrib={CFGRIB_OK}  dask={DASK_OK}  pyarrow={PYARROW_OK}")
print("Imports OK" if (XARRAY_OK and CFGRIB_OK and PYARROW_OK) else "Corrigez les imports manquants avant de continuer.")


xarray=True  cfgrib=True  dask=True  pyarrow=True
Imports OK


In [ ]:
DATA_ROOT       = Path("./data")
COPERNICUS_ROOT = Path("./copernicus")
CACHE_ROOT      = Path("./cache")
CACHE_ROOT.mkdir(exist_ok=True)
OUTPUT_VERSION_TAG = "v5"

AIRPORT_BOXES = {
    "LFPG": (  2.344072,  48.872047,  2.796588, 49.168166, "CDG"),
    "LFPO": (  2.193711,  48.627845,  2.555575, 48.858186, "ORY"),
    "LFMN": (  7.111992,  43.573432,  7.341670, 43.757419, "NCE"),
    "LFLL": (  4.974663,  45.596885,  5.213547, 45.848091, "LYS"),
    "LFML": (  5.061764,  43.324944,  5.394476, 43.559195, "MRS"),
    "LFBO": (  1.246603,  43.542128,  1.481490, 43.728042, "TLS"),
    "LFBD": ( -0.835362,  44.748425, -0.594478, 44.927034, "BOD"),
    "LFSB": (  7.389355,  47.512760,  7.679707, 47.706760, "MLH"),
}
IATA_TO_ICAO = {v[4]: k for k, v in AIRPORT_BOXES.items()}
FRANCE_LON_MIN, FRANCE_LON_MAX = -6.0, 11.0
FRANCE_LAT_MIN, FRANCE_LAT_MAX = 38.4, 51.7
WEATHER_GRID_MARGIN_DEG = 0.3

RUNWAY_ENDS = {
    "LFPG": [
        {"ident": "08L", "lat": 48.99570083618164, "lon": 2.5527400970458984, "heading_deg": 85.0},
        {"ident": "26R", "lat": 48.99879837036133, "lon": 2.610179901123047,  "heading_deg": 265.0},
        {"ident": "08R", "lat": 48.99290084838867, "lon": 2.565659999847412,  "heading_deg": 85.1},
        {"ident": "26L", "lat": 48.99489974975586, "lon": 2.6024301052093506, "heading_deg": 265.1},
        {"ident": "09L", "lat": 49.02470016479492, "lon": 2.5248899459838867, "heading_deg": 85.3},
        {"ident": "27R", "lat": 49.02669906616211, "lon": 2.561690092086792,  "heading_deg": 265.3},
        {"ident": "09R", "lat": 49.020599365234375,"lon": 2.5130600929260254,"heading_deg": 86.0},
        {"ident": "27L", "lat": 49.02370071411133, "lon": 2.5702900886535645, "heading_deg": 266.0},
    ],
    "LFPO": [
        {"ident": "02", "lat": 48.717498779296875, "lon": 2.376699924468994,  "heading_deg": 18.0},
        {"ident": "20", "lat": 48.737998962402344, "lon": 2.386970043182373,  "heading_deg": 198.0},
        {"ident": "06", "lat": 48.720001220703125, "lon": 2.316920042037964,  "heading_deg": 62.0},
        {"ident": "24", "lat": 48.73550033569336,  "lon": 2.360680103302002,  "heading_deg": 242.0},
        {"ident": "07", "lat": 48.719398498535156, "lon": 2.3585898876190186, "heading_deg": 74.0},
        {"ident": "25", "lat": 48.72740173339844,  "lon": 2.4020700454711914, "heading_deg": 254.0},
    ],
    "LFMN": [
        {"ident": "04L", "lat": 43.651798, "lon": 7.20404,  "heading_deg": 45.0},
        {"ident": "22R", "lat": 43.669498, "lon": 7.22851,  "heading_deg": 225.0},
        {"ident": "04R", "lat": 43.64670181274414, "lon": 7.202489852905273, "heading_deg": 45.0},
        {"ident": "22L", "lat": 43.66559982299805,  "lon": 7.228439807891846, "heading_deg": 225.0},
    ],
    "LFLL": [
        {"ident": "17L", "lat": 45.734901428222656, "lon": 5.091869831085205, "heading_deg": 175.0},
        {"ident": "35R", "lat": 45.71099853515625,  "lon": 5.094779968261719, "heading_deg": 355.0},
        {"ident": "17R", "lat": 45.74660110473633,  "lon": 5.085939884185791, "heading_deg": 175.0},
        {"ident": "35L", "lat": 45.71070098876953,  "lon": 5.0903000831604,   "heading_deg": 355.0},
    ],
    "LFML": [
        {"ident": "13L", "lat": 43.449100494384766, "lon": 5.197309970855713, "heading_deg": 133.7},
        {"ident": "31R", "lat": 43.42729949951172,  "lon": 5.22845983505249,  "heading_deg": 313.7},
        {"ident": "13R", "lat": 43.44129943847656,  "lon": 5.203020095825195, "heading_deg": 134.4},
        {"ident": "31L", "lat": 43.425899505615234, "lon": 5.2243499755859375,"heading_deg": 314.4},
    ],
    "LFBO": [
        {"ident": "14L", "lat": 43.63740158081055, "lon": 1.3576200008392334, "heading_deg": 143.0},
        {"ident": "32R", "lat": 43.6156005859375,  "lon": 1.3802200555801392, "heading_deg": 323.0},
        {"ident": "14R", "lat": 43.644100189208984,"lon": 1.3459299802780151, "heading_deg": 143.0},
        {"ident": "32L", "lat": 43.61899948120117, "lon": 1.3720999956130981, "heading_deg": 323.0},
    ],
    "LFBD": [
        {"ident": "05", "lat": 44.81909942626953, "lon": -0.7289829850196838, "heading_deg": 46.0},
        {"ident": "23", "lat": 44.83869934082031, "lon": -0.7009999752044678, "heading_deg": 226.0},
        {"ident": "11", "lat": 44.831600189208984,"lon": -0.7292420268058777, "heading_deg": 107.0},
        {"ident": "29", "lat": 44.825401306152344,"lon": -0.6998969912528992, "heading_deg": 287.0},
    ],
    "LFSB": [
        {"ident": "07", "lat": 47.587943, "lon": 7.516868, "heading_deg": 77.0},
        {"ident": "25", "lat": 47.591429, "lon": 7.539109, "heading_deg": 257.0},
        {"ident": "15", "lat": 47.617699, "lon": 7.509862, "heading_deg": 155.0},
        {"ident": "33", "lat": 47.585933, "lon": 7.531962, "heading_deg": 335.0},
    ],
}

VAR_LONGNAME_TO_SHORT_GUESS = {
    "Divergence":                              "d",
    "Fraction of cloud cover":                 "cc",
    "Geopotential":                            "z",
    "Ozone mass mixing ratio":                 "o3",
    "Potential vorticity":                     "pv",
    "Relative humidity":                       "r",
    "Specific cloud ice water content":        "ciwc",
    "Specific cloud liquid water content":     "clwc",
    "Specific humidity":                       "q",
    "Specific rain water content":             "crwc",
    "Specific snow water content":             "cswc",
    "Temperature":                             "t",
    "U component of wind":                     "u",
    "V component of wind":                     "v",
    "Vertical velocity":                       "w",
    "Vorticity (relative)":                    "vo",
}

def copernicus_path(d: date) -> Path:
    return COPERNICUS_ROOT / f"copernicus_{d.year}_{d.month:02d}_{d.day:02d}.grib"

copernicus_files = sorted(COPERNICUS_ROOT.glob("copernicus_*.grib"))
_CDATE_PAT = re.compile(r"copernicus_(\d{4})_(\d{2})_(\d{2})\.grib$")
STUDY_DATES = sorted({
    date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
    for p in copernicus_files if (m := _CDATE_PAT.search(p.name))
})
print(f"{len(STUDY_DATES)} journée(s) Copernicus détectée(s).")
if STUDY_DATES:
    print(f"De {STUDY_DATES[0]} à {STUDY_DATES[-1]}")
else:
    print("Aucun fichier copernicus_*.grib trouvé dans", COPERNICUS_ROOT.resolve())


24 journée(s) Copernicus détectée(s).
De 2019-02-25 à 2022-04-18


---
## 2 — Extraction météo :

In [ ]:
WEATHER_GRID_CACHE = CACHE_ROOT / f"weather_grid_by_airport_{OUTPUT_VERSION_TAG}.parquet"
N_JOBS = -1 

def _open_day_datasets(path: Path):
    idx_path = str(CACHE_ROOT / f"{path.name}.idx")
    backend_kwargs = {"indexpath": idx_path}
    chunk_opt = {"time": 1} if DASK_OK else None
    return cfgrib.open_datasets(str(path), backend_kwargs=backend_kwargs, chunks=chunk_opt)

def extract_all_airports_weather_grid(datasets, airport_boxes: dict, margin_deg: float) -> pd.DataFrame:
    per_airport_frames = {icao: [] for icao in airport_boxes}
    for ds_i in datasets:
        lat_name = "latitude" if "latitude" in ds_i.coords else "lat"
        lon_name = "longitude" if "longitude" in ds_i.coords else "lon"
        ds_i = ds_i.load()
        lat_vals = ds_i[lat_name].values
        descending_lat = lat_vals[0] > lat_vals[-1]

        for icao, box in airport_boxes.items():
            lon_min, lat_min, lon_max, lat_max, _ = box
            lon_min_m, lat_min_m = lon_min - margin_deg, lat_min - margin_deg
            lon_max_m, lat_max_m = lon_max + margin_deg, lat_max + margin_deg
            lat_slice = slice(lat_max_m, lat_min_m) if descending_lat else slice(lat_min_m, lat_max_m)
            try:
                sub = ds_i.sel({lat_name: lat_slice, lon_name: slice(lon_min_m, lon_max_m)})
            except Exception as e:
                print(f"sel() a échoué sur un hypercube pour {icao} ({e}) — ignoré.")
                continue
            if sub.sizes.get(lat_name, 0) == 0 or sub.sizes.get(lon_name, 0) == 0:
                continue
            if sub.sizes.get(lat_name, 0) < 2 or sub.sizes.get(lon_name, 0) < 2:
                print(f"Attention : {icao} n'a qu'un seul point de grille en lat/lon dans la marge "
                      f"({margin_deg}°) — augmente WEATHER_GRID_MARGIN_DEG pour permettre une vraie "
                      f"interpolation bilinéaire (sinon on retombe sur du plus-proche-voisin).")
            df_i = sub.to_dataframe().reset_index()
            per_airport_frames[icao].append(df_i)
    out_frames = []
    for icao, frames in per_airport_frames.items():
        if not frames:
            continue
        out = frames[0]
        for extra in frames[1:]:
            merge_cols = [c for c in ("time", "level", "isobaricInhPa", "step", "latitude", "longitude")
                          if c in out.columns and c in extra.columns]
            out = out.merge(extra, on=merge_cols, how="outer")
        out["airport_icao"] = icao
        out_frames.append(out)
    if not out_frames:
        return pd.DataFrame()
    return pd.concat(out_frames, ignore_index=True)

def _process_one_day(d: date) -> pd.DataFrame:
    path = copernicus_path(d)
    if not path.exists():
        print(f"fichier manquant pour {d}, ignoré.")
        return pd.DataFrame()
    print(f"Traitement {d} …")
    datasets = _open_day_datasets(path)
    df_day = extract_all_airports_weather_grid(datasets, AIRPORT_BOXES, WEATHER_GRID_MARGIN_DEG)
    if not df_day.empty:
        df_day["study_date"] = pd.Timestamp(d)
    del datasets
    gc.collect()
    return df_day

def build_or_load_weather_grid_cache(force_recompute: bool = False) -> pd.DataFrame:
    if WEATHER_GRID_CACHE.exists() and not force_recompute:
        print(f"Cache trouvé → chargement de {WEATHER_GRID_CACHE}")
        return pd.read_parquet(WEATHER_GRID_CACHE)
    try:
        from joblib import Parallel, delayed
        JOBLIB_OK = True
    except ImportError:
        JOBLIB_OK = False
        print("joblib non installé.")
    if JOBLIB_OK and N_JOBS != 1:
        results = Parallel(n_jobs=N_JOBS, prefer="processes")(delayed(_process_one_day)(d) for d in STUDY_DATES)
    else:
        results = [_process_one_day(d) for d in STUDY_DATES]

    all_rows = [df for df in results if not df.empty]
    if not all_rows:
        raise RuntimeError("Aucune donnée météo extraite — vérifiez COPERNICUS_ROOT et les journées disponibles.")
    weather_grid = pd.concat(all_rows, ignore_index=True)
    weather_grid.to_parquet(WEATHER_GRID_CACHE, index=False)
    print(f"Cache écrit → {WEATHER_GRID_CACHE} ({len(weather_grid):,} lignes)")
    return weather_grid

weather_grid = build_or_load_weather_grid_cache(force_recompute=False)


Cache écrit → cache\weather_grid_by_airport_v3.parquet (1,415,232 lignes)


---
## 3 — Feature engineering météo

In [ ]:
LEVEL_COL = "level" if "level" in weather_grid.columns else "isobaricInhPa"
G0 = 9.80665
wp = weather_grid.copy()
wp = wp.rename(columns={LEVEL_COL: "level_hPa"})

if {"u", "v"}.issubset(wp.columns):
    wp["wind_speed_ms"] = np.sqrt(wp["u"]**2 + wp["v"]**2)
    wp["wind_dir_deg"]  = (np.degrees(np.arctan2(-wp["u"], -wp["v"]))) % 360
if "z" in wp.columns:
    wp["height_m"] = wp["z"] / G0
water_cols = [c for c in ["ciwc", "clwc", "crwc", "cswc"] if c in wp.columns]
if water_cols:
    wp["convective_water_content"] = wp[water_cols].sum(axis=1)
wp = wp.sort_values(["airport_icao", "study_date", "time", "latitude", "longitude", "level_hPa"])

def _shear_and_lapse(g: pd.DataFrame) -> pd.Series:
    g = g.sort_values("level_hPa", ascending=False)
    if len(g) < 2 or "height_m" not in g.columns:
        return pd.Series({"wind_shear_ms_per_km": np.nan, "lapse_rate_K_per_km": np.nan})
    low, high = g.iloc[0], g.iloc[-1]
    dz_km = (high["height_m"] - low["height_m"]) / 1000
    if dz_km <= 0:
        return pd.Series({"wind_shear_ms_per_km": np.nan, "lapse_rate_K_per_km": np.nan})
    dwind = high.get("wind_speed_ms", np.nan) - low.get("wind_speed_ms", np.nan)
    dT    = high.get("t", np.nan) - low.get("t", np.nan)
    return pd.Series({"wind_shear_ms_per_km": dwind / dz_km, "lapse_rate_K_per_km": -dT / dz_km})

shear_lapse = (wp.groupby(["airport_icao", "study_date", "time", "latitude", "longitude"])
                 .apply(_shear_and_lapse).reset_index())

VARS_TO_MATCH = [c for c in ["wind_speed_ms", "wind_dir_deg", "t", "convective_water_content", "r"]
                  if c in wp.columns]
print(f"Variables météo disponibles pour l'appariement 4D : {VARS_TO_MATCH}")
print(f"Grille météo : {wp['airport_icao'].nunique()} aéroports, "
      f"{wp.groupby('airport_icao')[['latitude']].nunique()['latitude'].to_dict()} points de lat par aéroport.")


Variables météo disponibles pour l'appariement 4D : ['wind_speed_ms', 'wind_dir_deg', 't', 'convective_water_content', 'r']
Grille météo : 8 aéroports, {'LFBD': 3, 'LFBO': 4, 'LFLL': 3, 'LFML': 3, 'LFMN': 3, 'LFPG': 3, 'LFPO': 3, 'LFSB': 4} points de lat par aéroport.


---
## 4 — Chargement des trajectoires ADS-B

In [5]:
_DATE_PAT_ADSB = re.compile(r"data_(\d{4}-\d{2}-\d{2})-\d{2}\.parquet$")

def load_trajectories_for_dates(dates: list[date], data_root: Path) -> dict:
    wanted = {d.isoformat() for d in dates}
    all_parquet = sorted(data_root.glob("data_*.parquet"))
    parquet_files = [str(p) for p in all_parquet if (m := _DATE_PAT_ADSB.search(p.name)) and m.group(1) in wanted]
    if not parquet_files:
        print("Aucun fichier parquet ADS-B ne correspond à STUDY_DATES.")
        return {icao: pd.DataFrame() for icao in AIRPORT_BOXES}
    lf = (
        pl.scan_parquet(parquet_files)
        .rename({"time": "timestamp", "lat": "latitude", "lon": "longitude","velocity": "groundspeed", "heading": "track","vertrate": "vertical_rate", "baroaltitude": "altitude"})
        .filter((pl.col("longitude") >= FRANCE_LON_MIN) & (pl.col("longitude") <= FRANCE_LON_MAX) & (pl.col("latitude")  >= FRANCE_LAT_MIN) & (pl.col("latitude")  <= FRANCE_LAT_MAX))
        .drop_nulls(subset=["latitude", "longitude", "timestamp"])
    )
    print(f"Scan de {len(parquet_files)} fichier(s) parquet…")
    raw = lf.collect()
    print(f"  Lignes après filtre France : {len(raw):,}")
    if len(raw) == 0:
        return {icao: pd.DataFrame() for icao in AIRPORT_BOXES}
    raw = raw.with_columns([
        (pl.col("groundspeed") * 1.94384).alias("groundspeed"),
        (pl.col("altitude") * 3.28084).alias("altitude"),
        (pl.col("geoaltitude") * 3.28084).alias("geoaltitude") if "geoaltitude" in raw.columns else pl.lit(None).alias("geoaltitude"),
        (pl.col("vertical_rate") * 196.85).alias("vertical_rate"),
        (pl.col("timestamp").cast(pl.Int64) * 1_000).cast(pl.Datetime("ms")).dt.replace_time_zone("UTC").alias("timestamp"),
        pl.col("callsign").str.strip_chars().alias("callsign"),
    ])
    out = {}
    for icao, (lon_min, lat_min, lon_max, lat_max, _name) in AIRPORT_BOXES.items():
        sub = raw.filter((pl.col("longitude") >= lon_min) & (pl.col("longitude") <= lon_max) & (pl.col("latitude")  >= lat_min) & (pl.col("latitude")  <= lat_max))
        out[icao] = sub.to_pandas()
    return out

print("Chargement ADS-B pour les journées Copernicus disponibles…")
traffic_by_airport = load_trajectories_for_dates(STUDY_DATES, DATA_ROOT)
for icao, df in traffic_by_airport.items():
    print(f"  {icao} ({AIRPORT_BOXES[icao][4]}): {len(df):,} points")


Chargement ADS-B pour les journées Copernicus disponibles…
Scan de 576 fichier(s) parquet…
  Lignes après filtre France : 97,680,203
  LFPG (CDG): 1,504,175 points
  LFPO (ORY): 1,063,172 points
  LFMN (NCE): 101,847 points
  LFLL (LYS): 115,923 points
  LFML (MRS): 157,782 points
  LFBO (TLS): 480,313 points
  LFBD (BOD): 92,045 points
  LFSB (MLH): 364,101 points


---
## 5 — Détection d'approche finale + affectation de piste

In [ ]:
APPROACH_MIN_POINTS = 8
APPROACH_MAX_ALTITUDE_FT = 10_000
APPROACH_MAX_GROUNDSPEED_KT = 320
APPROACH_MAX_DISTANCE_KM = 30
GO_AROUND_ALT_FLOOR_FT = 2500
GO_AROUND_CLIMB_FT = 800
FINAL_SEGMENT_ALONG_MAX_KM = 8.0 
FINAL_SEGMENT_ALONG_MIN_KM = 0.3   
MAX_CROSSTRACK_RMS_KM = 4.0  
MAX_HEADING_MISALIGN_DEG = 45.0
MIN_FINAL_SEGMENT_POINTS = 3

def airport_reference_point(airport_icao):
    lon_min, lat_min, lon_max, lat_max, _ = AIRPORT_BOXES[airport_icao]
    return (lat_min + lat_max) / 2, (lon_min + lon_max) / 2

def haversine_km(lat1, lon1, lat2, lon2):
    r_earth = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * r_earth * np.arcsin(np.sqrt(a))

def extract_airline_from_callsign(callsign):
    if pd.isna(callsign):
        return "UNK"
    code_ = re.sub(r"[^A-Z0-9]", "", str(callsign).strip().upper())
    m = re.match(r"([A-Z]{2,3})", code_)
    return m.group(1) if m else "UNK"

def circular_diff(a, b):
    return np.abs((np.asarray(a, dtype=float) - np.asarray(b, dtype=float) + 180) % 360 - 180)

def runway_relative_coords(lat, lon, rwy_lat, rwy_lon, rwy_heading_deg):
    lat0 = np.radians(rwy_lat)
    dx_east_km  = (np.asarray(lon) - rwy_lon) * 111.320 * np.cos(lat0)
    dy_north_km = (np.asarray(lat) - rwy_lat) * 110.574
    hdg = np.radians(rwy_heading_deg)
    ux, uy = np.sin(hdg), np.cos(hdg)
    along_km =  dx_east_km * ux + dy_north_km * uy
    cross_km = -dx_east_km * uy + dy_north_km * ux
    return along_km, cross_km

def add_approach_geometry(df, airport_icao):
    out = df.copy()
    airport_lat, airport_lon = airport_reference_point(airport_icao)
    out["distance_to_airport_km"] = haversine_km(out["latitude"], out["longitude"], airport_lat, airport_lon)
    return out

def split_candidate_flights(df):
    out = df.sort_values(["icao24", "callsign", "timestamp"]).copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True, errors="coerce")
    gap_min = out.groupby(["icao24", "callsign"], dropna=False)["timestamp"].diff().dt.total_seconds().div(60)
    new_segment = gap_min.isna() | (gap_min > 20)
    out["segment_id"] = new_segment.groupby([out["icao24"].fillna(""), out["callsign"].fillna("")], dropna=False).cumsum().astype(int)
    out["approach_id"] = out["icao24"].fillna("UNK").astype(str) + "_" + out["callsign"].fillna("UNK").astype(str) + "_" + out["segment_id"].astype(str)
    return out

def detect_go_around(g: pd.DataFrame, altitude_col: str) -> bool:
    g = g.sort_values("timestamp")
    if "onground" in g.columns and g["onground"].fillna(False).any():
        return False
    alt = g[altitude_col].values
    below = np.where(alt <= GO_AROUND_ALT_FLOOR_FT)[0]
    if len(below) == 0:
        return False
    low_idx = below[np.argmin(alt[below])]
    after = alt[low_idx:]
    if len(after) < 2:
        return False
    return bool((after.max() - after[0]) >= GO_AROUND_CLIMB_FT)

def assign_runway(g: pd.DataFrame, airport_icao: str):
    candidates = RUNWAY_ENDS.get(airport_icao, [])
    if not candidates or "track" not in g.columns:
        return None, None, None
    final_track = g["track"].tail(3).mean()
    if pd.isna(final_track):
        return None, None, None

    best = None
    best_rms = np.inf
    best_along = best_cross = None
    for rwy in candidates:
        if circular_diff(final_track, rwy["heading_deg"]) > MAX_HEADING_MISALIGN_DEG:
            continue
        along_km, cross_km = runway_relative_coords(
            g["latitude"].values, g["longitude"].values, rwy["lat"], rwy["lon"], rwy["heading_deg"])
        mask_final = (along_km <= -FINAL_SEGMENT_ALONG_MIN_KM) & (along_km >= -FINAL_SEGMENT_ALONG_MAX_KM)
        if mask_final.sum() < MIN_FINAL_SEGMENT_POINTS:
            continue
        idx_final = np.where(mask_final)[0]
        if not np.all(np.diff(along_km[idx_final]) >= -0.05): 
            trend_ok = np.corrcoef(idx_final, along_km[idx_final])[0, 1] > 0.3 
            if not trend_ok:
                continue
        rms_cross = np.sqrt(np.mean(cross_km[mask_final] ** 2))
        if rms_cross > MAX_CROSSTRACK_RMS_KM:
            continue
        if rms_cross < best_rms:
            best_rms = rms_cross
            best = rwy
            best_along, best_cross = along_km, cross_km
    return best, best_along, best_cross

def detect_final_approaches_for_airport(df, study_date, airport_icao):
    required = {"timestamp", "latitude", "longitude", "icao24", "callsign", "groundspeed", "vertical_rate", "track"}
    if df.empty or not required.issubset(df.columns):
        return pd.DataFrame()
    work = add_approach_geometry(df, airport_icao)
    altitude_col = "geoaltitude" if "geoaltitude" in work.columns and work["geoaltitude"].notna().any() else "altitude"
    work = work.dropna(subset=["timestamp", "latitude", "longitude", "distance_to_airport_km"]).copy()
    work = work[(work["distance_to_airport_km"] <= APPROACH_MAX_DISTANCE_KM) &
                (work[altitude_col].fillna(np.inf) <= APPROACH_MAX_ALTITUDE_FT) &
                (work["groundspeed"].fillna(np.inf) <= APPROACH_MAX_GROUNDSPEED_KT)].copy()
    if work.empty:
        return pd.DataFrame()
    work = split_candidate_flights(work)
    detected = []
    n_rejected_no_runway = 0
    for approach_id, g in work.groupby("approach_id", sort=False):
        g = g.sort_values("timestamp").copy()
        if len(g) < APPROACH_MIN_POINTS:
            continue
        rwy, along_km, cross_km = assign_runway(g, airport_icao)
        if rwy is None:
            n_rejected_no_runway += 1
            continue
        g["runway_ident"] = rwy["ident"]
        g["runway_heading_deg"] = rwy["heading_deg"]
        g["along_track_km"] = along_km
        g["distance_to_threshold_km"] = -along_km
        g["cross_track_km"] = cross_km

        g["distance_band_km"] = pd.cut(g["distance_to_threshold_km"], bins=[0, 5, 10, 20, 30],
                                        labels=["<=5 km", "5-10 km", "10-20 km", "20-30 km"], include_lowest=True)
        g["study_date"] = pd.Timestamp(study_date)
        g["airport_icao"] = airport_icao
        g["airport"] = AIRPORT_BOXES[airport_icao][4]
        g["approach_id"] = f"{study_date}_{airport_icao}_{approach_id}"
        g["company"] = g["callsign"].apply(extract_airline_from_callsign)
        g["go_around"] = detect_go_around(g, altitude_col)
        detected.append(g)
    if n_rejected_no_runway:
        print(f"  {airport_icao} {study_date} : {n_rejected_no_runway} segment(s) rejeté(s) "
              f"(aucune piste alignée/stable trouvée).")
    return pd.concat(detected, ignore_index=True) if detected else pd.DataFrame()

approach_frames = []
for study_date in STUDY_DATES:
    for airport_icao, df_airport in traffic_by_airport.items():
        if df_airport.empty:
            continue
        day_df = df_airport[df_airport["timestamp"].dt.date == study_date]
        detected = detect_final_approaches_for_airport(day_df, study_date, airport_icao)
        if not detected.empty:
            approach_frames.append(detected)

approach_points = pd.concat(approach_frames, ignore_index=True) if approach_frames else pd.DataFrame()
if approach_points.empty:
    raise ValueError("Aucune approche détectée — vérifiez STUDY_DATES / AIRPORT_BOXES / colonnes disponibles.")
print(f"Points d'approche (grain le plus fin, une ligne par point ADS-B) : {len(approach_points):,}")
print(f"Approches distinctes détectées : {approach_points['approach_id'].nunique():,} "
      f"(dont {approach_points.groupby('approach_id')['go_around'].first().sum()} go-around)")
print(approach_points['runway_ident'].value_counts())


  LFPG 2019-02-25 : 633 segment(s) rejeté(s) (aucune piste alignée/stable trouvée).
  LFPO 2019-02-25 : 285 segment(s) rejeté(s) (aucune piste alignée/stable trouvée).
  LFLL 2019-02-25 : 100 segment(s) rejeté(s) (aucune piste alignée/stable trouvée).
  LFML 2019-02-25 : 98 segment(s) rejeté(s) (aucune piste alignée/stable trouvée).
  LFBO 2019-02-25 : 113 segment(s) rejeté(s) (aucune piste alignée/stable trouvée).
  LFBD 2019-02-25 : 61 segment(s) rejeté(s) (aucune piste alignée/stable trouvée).
  LFSB 2019-02-25 : 86 segment(s) rejeté(s) (aucune piste alignée/stable trouvée).
  LFPG 2019-03-04 : 606 segment(s) rejeté(s) (aucune piste alignée/stable trouvée).
  LFPO 2019-03-04 : 268 segment(s) rejeté(s) (aucune piste alignée/stable trouvée).
  LFMN 2019-03-04 : 6 segment(s) rejeté(s) (aucune piste alignée/stable trouvée).
  LFLL 2019-03-04 : 97 segment(s) rejeté(s) (aucune piste alignée/stable trouvée).
  LFML 2019-03-04 : 118 segment(s) rejeté(s) (aucune piste alignée/stable trouvée)

---
## 6 — Appariement météo 4D, point par point

In [ ]:
from scipy.interpolate import RegularGridInterpolator
VARS_4D = VARS_TO_MATCH + (["height_m"] if "height_m" not in VARS_TO_MATCH else [])

def _epoch_ns(series_like) -> np.ndarray:
    idx = pd.DatetimeIndex(pd.to_datetime(series_like))
    if idx.tz is not None:
        idx = idx.tz_convert("UTC").tz_localize(None)
    try:
        return idx.as_unit("ns").asi8 
    except AttributeError:
        return idx.astype("datetime64[ns]").astype("int64").values 

def _build_grid_interpolators(g_wp: pd.DataFrame, variables: list[str]):
    time_int = pd.Series(_epoch_ns(g_wp["time"]), index=g_wp.index) 
    times_sorted = np.sort(time_int.unique())
    levels_sorted = np.sort(g_wp["level_hPa"].unique())    
    lats_sorted = np.sort(g_wp["latitude"].unique())
    lons_sorted = np.sort(g_wp["longitude"].unique())
    if min(len(times_sorted), len(levels_sorted), len(lats_sorted), len(lons_sorted)) < 1:
        return None
    time_num = (times_sorted - times_sorted[0]) / 1e9 

    t_idx = {v: i for i, v in enumerate(times_sorted)}
    l_idx = {v: i for i, v in enumerate(levels_sorted)}
    la_idx = {v: i for i, v in enumerate(lats_sorted)}
    lo_idx = {v: i for i, v in enumerate(lons_sorted)}

    shape = (len(times_sorted), len(levels_sorted), len(lats_sorted), len(lons_sorted))
    interpolators = {}
    for var in variables:
        if var not in g_wp.columns:
            continue
        arr = np.full(shape, np.nan)
        ti = time_int.map(t_idx).values
        li = g_wp["level_hPa"].map(l_idx).values
        lai = g_wp["latitude"].map(la_idx).values
        loi = g_wp["longitude"].map(lo_idx).values
        valid = ~(pd.isna(ti) | pd.isna(li) | pd.isna(lai) | pd.isna(loi))
        arr[ti[valid].astype(int), li[valid].astype(int), lai[valid].astype(int), loi[valid].astype(int)] = \
            g_wp[var].values[valid]
        if np.isnan(arr).all():
            continue
        if np.isnan(arr).any():
            arr = pd.DataFrame(arr.reshape(shape[0]*shape[1], -1)).ffill().bfill().values.reshape(shape)
        interpolators[var] = RegularGridInterpolator(
            (time_num, levels_sorted, lats_sorted, lons_sorted), arr,
            method="linear", bounds_error=False, fill_value=None)
    return {
        "interpolators": interpolators,
        "time_ref_int_ns": times_sorted[0],
        "levels": levels_sorted,
        "lat_bounds": (lats_sorted.min(), lats_sorted.max()),
        "lon_bounds": (lons_sorted.min(), lons_sorted.max()),
        "time_bounds": (time_num.min(), time_num.max()),
    }

def match_weather_4d_for_group(points: pd.DataFrame, grid_bundle: dict) -> pd.DataFrame:
    n_pts = len(points)
    levels = grid_bundle["levels"]
    n_lvl = len(levels)
    time_ref_int_ns = grid_bundle["time_ref_int_ns"]
    time_num_pts = (_epoch_ns(points["timestamp"]) - time_ref_int_ns) / 1e9  # CORRECTIF unité
    lat_pts = points["latitude"].values
    lon_pts = points["longitude"].values

    lat_lo, lat_hi = grid_bundle["lat_bounds"]
    lon_lo, lon_hi = grid_bundle["lon_bounds"]
    t_lo, t_hi = grid_bundle["time_bounds"]
    extrapolated = ((lat_pts < lat_lo) | (lat_pts > lat_hi) |
                    (lon_pts < lon_lo) | (lon_pts > lon_hi) |
                    (time_num_pts < t_lo) | (time_num_pts > t_hi))

    t_rep = np.repeat(time_num_pts, n_lvl)
    lat_rep = np.repeat(lat_pts, n_lvl)
    lon_rep = np.repeat(lon_pts, n_lvl)
    lvl_tile = np.tile(levels, n_pts)
    query = np.column_stack([t_rep, lvl_tile, lat_rep, lon_rep])

    profiles = {}
    for var, interp in grid_bundle["interpolators"].items():
        profiles[var] = interp(query).reshape(n_pts, n_lvl)

    if "height_m" not in profiles:
        raise RuntimeError("height_m manquant : impossible d'interpoler verticalement (vérifier "
                            "que 'z' (géopotentiel) est bien présent dans les GRIB).")
    height_profiles = profiles["height_m"]
    altitude_m = points["_altitude_ft_for_match"].values * 0.3048

    out = {"weather_extrapolated": extrapolated}
    for var in profiles:
        if var == "height_m":
            continue
        vals = np.full(n_pts, np.nan)
        for i in range(n_pts):
            h = height_profiles[i]
            v = profiles[var][i]
            order = np.argsort(h)
            vals[i] = np.interp(altitude_m[i], h[order], v[order])
        out[f"{var}_pt"] = vals
    if "wind_speed_ms" in profiles:
        shear_vals = np.full(n_pts, np.nan)
        for i in range(n_pts):
            h = height_profiles[i]
            v = profiles["wind_speed_ms"][i]
            order = np.argsort(h)
            h_sorted, v_sorted = h[order], v[order]
            grad_ms_per_m = np.gradient(v_sorted, h_sorted)
            shear_vals[i] = np.interp(altitude_m[i], h_sorted, grad_ms_per_m) * 1000 
        out["wind_shear_ms_per_km_pt"] = shear_vals

    return pd.DataFrame(out, index=points.index)

altitude_col_match = "geoaltitude" if "geoaltitude" in approach_points.columns and approach_points["geoaltitude"].notna().any() else "altitude"
approach_points["_altitude_ft_for_match"] = approach_points[altitude_col_match]

matched_chunks = []
for (icao, sdate), pts in approach_points.groupby(["airport_icao", "study_date"], sort=False):
    g_wp = wp[(wp["airport_icao"] == icao) & (wp["study_date"] == sdate)]
    if g_wp.empty:
        print(f"Pas de données météo pour {icao} / {sdate} — points non appariés.")
        continue
    bundle = _build_grid_interpolators(g_wp, VARS_4D)
    if bundle is None or not bundle["interpolators"]:
        print(f"Grille météo insuffisante pour {icao} / {sdate}.")
        continue
    matched = match_weather_4d_for_group(pts, bundle)
    matched_chunks.append(matched)

weather_matched = pd.concat(matched_chunks) if matched_chunks else pd.DataFrame()
approach_points_weather = approach_points.join(weather_matched, how="left")

match_rate = approach_points_weather.get("wind_speed_ms_pt", pd.Series(dtype=float)).notna().mean()
print(f"Appariement 4D réussi pour {match_rate:.1%} des points d'approche.")
extrap_rate = approach_points_weather.get("weather_extrapolated", pd.Series(dtype=float)).mean()
print(f"Part des points hors domaine de la grille météo (extrapolés) : {extrap_rate:.1%} "
      f"(si élevé, augmente WEATHER_GRID_MARGIN_DEG et regénère le cache météo).")
if {"wind_speed_ms_pt", "wind_dir_deg_pt", "runway_heading_deg"}.issubset(approach_points_weather.columns):
    rel_angle = np.radians(approach_points_weather["wind_dir_deg_pt"] - approach_points_weather["runway_heading_deg"])
    approach_points_weather["headwind_ms"]  = approach_points_weather["wind_speed_ms_pt"] * np.cos(rel_angle)
    approach_points_weather["crosswind_ms"] = approach_points_weather["wind_speed_ms_pt"] * np.sin(rel_angle)


Appariement 4D réussi pour 100.0% des points d'approche.
Part des points hors domaine de la grille météo (extrapolés) : 0.8% (si élevé, augmente WEATHER_GRID_MARGIN_DEG et regénère le cache météo).


---
## 7 — Assemblage final, contrôle qualité, export

In [ ]:
FINAL_OUTPUT_PATH = CACHE_ROOT / f"trajectoires_meteo_points_{OUTPUT_VERSION_TAG}.parquet"

ID_COLS = ["approach_id", "airport_icao", "airport", "study_date", "company", "go_around",
           "runway_ident", "runway_heading_deg"]
TIME_GEOM_COLS = ["timestamp", "latitude", "longitude", altitude_col_match, "groundspeed",
                  "vertical_rate", "track", "along_track_km", "distance_to_threshold_km",
                  "cross_track_km", "distance_band_km"]
WEATHER_COLS = [c for c in approach_points_weather.columns if c.endswith("_pt")] + \
               ["headwind_ms", "crosswind_ms", "weather_extrapolated"]

keep_cols = [c for c in ID_COLS + TIME_GEOM_COLS + WEATHER_COLS if c in approach_points_weather.columns]
final_points = approach_points_weather[keep_cols].copy()
final_points = final_points.rename(columns={altitude_col_match: "altitude_ft"})

print("=== Contrôle qualité du fichier de jointure final ===")
print(f"Lignes (points ADS-B) : {len(final_points):,}")
print(f"Approches distinctes  : {final_points['approach_id'].nunique():,}")
print(f"Aéroports             : {sorted(final_points['airport_icao'].unique())}")
print(f"Taux de couverture météo (vent) : {final_points.get('wind_speed_ms_pt', pd.Series(dtype=float)).notna().mean():.1%}")
print(f"Taux d'extrapolation météo (hors grille) : {final_points['weather_extrapolated'].mean():.1%}")
print(f"Cross-track (km) — résumé :")
print(final_points["cross_track_km"].describe())
n_dupli = final_points.duplicated(subset=["approach_id", "timestamp"]).sum()
if n_dupli:
    print(f"ATTENTION : {n_dupli} doublons (approach_id, timestamp) détectés — à investiguer avant modélisation R.")

final_points.to_parquet(FINAL_OUTPUT_PATH, index=False, compression="zstd", engine="pyarrow")
size_mb = FINAL_OUTPUT_PATH.stat().st_size / 1e6
print(f"\nFichier final écrit : {FINAL_OUTPUT_PATH}  ({size_mb:.1f} Mo, {len(final_points):,} lignes)")
print("Lisible sous R avec :  arrow::read_parquet(\"trajectoires_meteo_points.parquet\")")


=== Contrôle qualité du fichier de jointure final ===
Lignes (points ADS-B) : 614,938
Approches distinctes  : 20,998
Aéroports             : ['LFBD', 'LFBO', 'LFLL', 'LFML', 'LFMN', 'LFPG', 'LFPO', 'LFSB']
Taux de couverture météo (vent) : 100.0%
Taux d'extrapolation météo (hors grille) : 0.8%
Cross-track (km) — résumé :
count    614938.000000
mean          0.066324
std           1.739723
min         -19.156259
25%          -0.005609
50%           0.003853
75%           0.022560
max          20.663566
Name: cross_track_km, dtype: float64

Fichier final écrit : cache\trajectoires_meteo_points_v3.parquet  (51.3 Mo, 614,938 lignes)
Lisible sous R avec :  arrow::read_parquet("trajectoires_meteo_points.parquet")
